In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data, ClusterData, ClusterLoader
from torch_geometric.nn import SAGEConv

In [4]:
class NormedLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(NormedLinear, self).__init__()
        self.weight = nn.Parameter(torch.Tensor(in_features, out_features))
        self.weight.data.uniform_(-1, 1).renorm_(2, 1, 1e-5).mul_(1e5)
    def forward(self, x):
        out = F.normalize(x, dim=1).mm(F.normalize(self.weight, dim=0))
        return 10 * out

class Encoder(nn.Module):
    """GraphSAGE Encoder"""
    def __init__(self, x_dim, num_cls, hid_dim=128):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(x_dim, hid_dim)
        self.conv = SAGEConv(hid_dim, hid_dim)
        self.relu = nn.ReLU()
        self.classifier = NormedLinear(hid_dim, num_cls)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.relu(self.fc1(x))
        x = self.relu(self.conv(x, edge_index))
        out = self.classifier(x)
        return out

In [5]:
import anndata
adata = anndata.read_h5ad("/mnt/jwh83-data/Confetti/output/Redsea/Clustering/mesmer_afterREDSEA_leiden.h5ad")
adata

/home/labuser/anaconda3/envs/ns_env/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


AnnData object with n_obs × n_vars = 161867 × 45
    obs: 'CD123', 'CDX2', 'slide_name', 'CellID', 'Unnamed: 0', 'cell_size', 'x_centroid', 'y_centroid', 'FILE', 'MUC6', 'GATA3', 'Lefty', 'CD279', 'NKG2D', 'CK7', 'PGP95', 'CD154', 'Somatostatin', 'CD294', 'DRAQ5', 'Hoechst1', 'leiden'
    uns: 'leiden', 'neighbors', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [6]:
def adata_to_dataframe(adata):
    """
    Convert AnnData object to DataFrame.

    Parameters:
        adata (AnnData): AnnData object to be converted.

    Returns:
        DataFrame: DataFrame containing both variables and observation attributes.
    """
    # Convert AnnData variables to DataFrame
    df_var = adata.to_df()

    # Extract observation attributes from AnnData
    df_obs = adata.obs

    # Concatenate variable DataFrame and observation attribute DataFrame along axis 1
    df = pd.concat([df_var, df_obs], axis=1)

    return df

In [7]:
adata_df = adata_to_dataframe(adata)
adata_df

,CD45RO,CD56,CD15,CD163,MUC2,CD34,CD36,aDefensin5,CD49a,HLA-DR,...,CD279,NKG2D,CK7,PGP95,CD154,Somatostatin,CD294,DRAQ5,Hoechst1,leiden
0,-0.792438,-0.464128,-0.404866,-0.569378,-0.232205,-0.508904,-0.394208,-0.188853,-0.534180,-0.655518,...,-0.520177,-0.287984,-0.285174,-0.370329,-0.237003,-0.396684,-0.366677,108.555901,1769.096273,26
1,-0.761787,-0.368181,-0.404866,-0.573279,-0.146500,0.093879,-0.203539,-0.188147,-0.884761,-0.565706,...,-0.481888,-0.263431,-0.278894,-0.332178,-0.229985,-0.372060,-0.361777,153.453237,1800.270983,2
2,-0.651427,-0.424421,-0.404866,-0.563891,-0.264746,-0.528378,-0.394208,-0.188686,0.719651,-0.603110,...,-0.508631,-0.289298,-0.285061,-0.320872,-0.235740,-0.340551,-0.357385,124.168880,1160.220114,30
3,-0.623663,-0.394461,-0.404866,-0.542871,-0.070682,-0.457703,-0.388858,-0.188787,2.030606,-0.588407,...,-0.501124,-0.282034,-0.284593,-0.303987,-0.234562,-0.327229,-0.350439,130.338480,1277.517815,30
4,-0.508038,-0.352165,-0.404866,-0.509874,-0.074431,-0.429613,-0.360887,-0.183052,0.340707,-0.577586,...,-0.491249,-0.264404,-0.272554,-0.259779,-0.212381,-0.240661,-0.331837,349.145078,1876.870466,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161862,0.254471,-0.433031,-0.404866,-0.452324,-0.318640,-0.251299,-0.349643,-0.188227,-0.818403,-0.392401,...,0.009324,-0.238160,-0.210302,-0.280664,-0.233341,-0.166325,-0.277777,1939.201043,8108.656323,7
161863,0.284696,-0.433441,-0.404866,-0.573279,-0.160261,-0.240392,-0.348079,-0.187067,-0.442319,-0.452224,...,0.107091,-0.246689,-0.194560,-0.263957,-0.233811,-0.174309,-0.194359,2031.516066,8344.850710,7
161864,0.077092,-0.464128,-0.404866,-0.547314,-0.131454,-0.290974,-0.368028,-0.188853,-0.870814,-0.477960,...,0.128706,-0.243232,-0.133573,-0.280254,-0.233356,-0.206683,-0.211327,878.046975,3685.936571,7
161865,-0.169521,-0.464128,-0.404866,-0.503331,-0.342721,-0.528378,-0.394208,-0.181601,-0.884761,-0.412729,...,-0.538261,-0.242060,0.132175,-0.319700,-0.227124,-0.265286,-0.360624,1353.259259,4338.037037,14


In [8]:
adata_df.columns

Index(['CD45RO', 'CD56', 'CD15', 'CD163', 'MUC2', 'CD34', 'CD36', 'aDefensin5',
       'CD49a', 'HLA-DR', 'CD38', 'CollagenIV', 'CD4', 'CD138', 'CD44',
       'Vimentin', 'CD66', 'Podoplanin', 'CHGA', 'CD3', 'SOX9', 'CD161',
       'Synaptophysin', 'CD57', 'ITLN1', 'CD127', 'CD45', 'CD49f', 'aSMA',
       'Cytokeratin', 'CD8', 'CD19', 'BCL2', 'CD90', 'CD21', 'CD7', 'CD11c',
       'Ki67', 'CD68', 'MUC1', 'CD206', 'CD16', 'CD117', 'CD69', 'CD31',
       'CD123', 'CDX2', 'slide_name', 'CellID', 'Unnamed: 0', 'cell_size',
       'x_centroid', 'y_centroid', 'FILE', 'MUC6', 'GATA3', 'Lefty', 'CD279',
       'NKG2D', 'CK7', 'PGP95', 'CD154', 'Somatostatin', 'CD294', 'DRAQ5',
       'Hoechst1', 'leiden'],
      dtype='object')

In [9]:
adata_df.columns.get_loc('CD123')

45

In [10]:
# create combined region identifier
#adata_df["unique_region"] = (
#    adata_df["sample_ID"].astype(str) + "_" + adata_df["segmentation_method"].astype(str)
#)

In [11]:
adata_df.to_csv("/mnt/jwh83-data/Confetti/output/Redsea/Clustering/20260119_ns444_mesmer_afterREDSEA_leiden.csv", index=False)

In [12]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix

# -------------------------------------------------------
# 0) Paths + config (EDIT)
# -------------------------------------------------------
TRAIN_CSV = "/mnt/jwh83-data/Confetti/output/Redsea/STELLAR/20251007_cleaned_trainingdata_yang.csv"  # used to reconstruct label_map + feature cols
MODEL_PT  = "/mnt/jwh83-data/Confetti/output/Redsea/STELLAR/fcnet_minigraphs_region_split_hubmapallregion.pt"

NEW_CSV   = "/mnt/jwh83-data/Confetti/output/Redsea/Clustering/20260119_ns444_mesmer_afterREDSEA_leiden.csv"
OUT_PREDS = "/mnt/jwh83-data/Confetti/output/Redsea/Clustering/20260119_ns444_mesmer_afterREDSEA_gnn_predict.csv"

# graph build params (MUST match your preprocessing)
marker_cols    = 45
coord_cols     = ('x_centroid', 'y_centroid')
label_col      = "cell_type_update"
region_col     = 'slide_name'
distance_thres = 100
cluster_size   = 300
sample_rate    = 1.0
seed           = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------------------------------
# 1) Reconstruct training artifacts (feature cols + label_map)
#    IMPORTANT: label_map order must match training
# -------------------------------------------------------
train_df = pd.read_csv(TRAIN_CSV)

# features are first marker_cols columns (same as your build_mini_graphs)
feature_cols = train_df.columns[:marker_cols].tolist()

y_str_train = train_df[label_col].astype(str).values if label_col in train_df.columns else train_df.iloc[:, 0].astype(str).values
label_names = np.sort(np.unique(y_str_train))
label_map = {n: i for i, n in enumerate(label_names)}
num_cls = len(label_map)
in_dim = len(feature_cols)

print(f"[Training artifacts] in_dim={in_dim}, num_cls={num_cls}")

# -------------------------------------------------------
# 2) Build mini-graphs for NEW dataset, BUT keep mapping to original row indices
#    (This is a graph-builder variant for inference.)
# -------------------------------------------------------
from sklearn.metrics import pairwise_distances
from torch_geometric.data import Data

def build_mini_graphs_for_inference(
    csv_path,
    feature_cols,
    marker_cols=44,
    coord_cols=('x', 'y'),
    region_col=None,
    distance_thres=100.0,
    cluster_size=300,
    sample_rate=1.0,
    random_state=42,
):
    df = pd.read_csv(csv_path)
    print(f"[NEW] Total cells: {len(df)}")

    # keep original row id for writing predictions back
    df["_row_id_"] = np.arange(len(df), dtype=np.int64)

    # optional sampling (keeps subset only)
    if sample_rate < 1.0:
        df = df.sample(frac=sample_rate, random_state=random_state).reset_index(drop=True)
        print(f"[NEW] Sampled {len(df)} cells ({sample_rate*100:.1f}%)")

    # ensure all feature cols exist; fill missing with 0.0
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0.0

    X_all = df[feature_cols].to_numpy(dtype=np.float32)
    pos_all = df.loc[:, list(coord_cols)].to_numpy(dtype=np.float32)
    row_ids = df["_row_id_"].to_numpy(dtype=np.int64)

    if region_col is None:
        df["_REGION_"] = "ALL"
        region_col_use = "_REGION_"
    else:
        if region_col not in df.columns:
            raise ValueError(f"region_col='{region_col}' not found in NEW CSV columns.")
        region_col_use = region_col

    regions = df[region_col_use].astype(str).values
    unique_regions = np.unique(regions)
    rng = np.random.RandomState(random_state)

    graphs = []
    for reg in unique_regions:
        reg_mask = (regions == reg)
        reg_idx = np.where(reg_mask)[0]
        if len(reg_idx) == 0:
            continue

        reg_idx = reg_idx.copy()
        rng.shuffle(reg_idx)

        num_clusters = int(np.ceil(len(reg_idx) / cluster_size))
        for i in range(num_clusters):
            sub_local = reg_idx[i*cluster_size : (i+1)*cluster_size]
            if len(sub_local) == 0:
                continue

            X_sub = X_all[sub_local]
            pos_sub = pos_all[sub_local]
            row_sub = row_ids[sub_local]   # mapping to original NEW df row indices

            dists = pairwise_distances(pos_sub)
            dmask = dists < distance_thres
            np.fill_diagonal(dmask, 0)

            edges = np.transpose(np.nonzero(dmask))
            edge_index = (
                torch.LongTensor(edges).T
                if len(edges) > 0
                else torch.empty((2, 0), dtype=torch.long)
            )

            g = Data(
                x=torch.FloatTensor(X_sub),
                edge_index=edge_index,
            )
            g.region = reg
            g.row_id = torch.LongTensor(row_sub)   # <-- critical for stitching predictions back
            graphs.append(g)

    print(f"[NEW] Built {len(graphs)} mini-graphs across {len(unique_regions)} region(s)")
    return df, graphs

new_df_full, graphs_new = build_mini_graphs_for_inference(
    NEW_CSV,
    feature_cols=feature_cols,
    marker_cols=marker_cols,
    coord_cols=coord_cols,
    region_col=region_col,
    distance_thres=distance_thres,
    cluster_size=cluster_size,
    sample_rate=sample_rate,
    random_state=seed,
)

# -------------------------------------------------------
# 3) Load model weights (same Encoder as training)
# -------------------------------------------------------
# You must have Encoder defined exactly as in training.
# Example:
# model = Encoder(in_dim, num_cls).to(device)

model = Encoder(in_dim, num_cls).to(device)

ckpt = torch.load(MODEL_PT, map_location=device)
# GraphBatchTrainer saved {"model_state": ...}
model.load_state_dict(ckpt["model_state"])
model.eval()

print("[Model] Loaded weights and set to eval()")

# -------------------------------------------------------
# 4) Inference: predict node labels for every graph, stitch back by row_id
# -------------------------------------------------------
N_total = len(pd.read_csv(NEW_CSV))
pred_id = np.full(N_total, -1, dtype=np.int64)
pred_conf = np.full(N_total, np.nan, dtype=np.float32)

with torch.no_grad():
    for g in graphs_new:
        g = g.to(device)
        logits = model(g)                    # (n_nodes, C)
        probs = F.softmax(logits, dim=1)
        ids = probs.argmax(dim=1).cpu().numpy()
        conf = probs.max(dim=1).values.cpu().numpy()

        rows = g.row_id.cpu().numpy()
        pred_id[rows] = ids
        pred_conf[rows] = conf

pred_name = np.array([label_names[i] if i >= 0 else "NA" for i in pred_id], dtype=object)

# -------------------------------------------------------
# 5) Save CSV with predictions
# -------------------------------------------------------
out_df = pd.read_csv(NEW_CSV)
out_df["pred_id"] = pred_id
out_df["pred_name"] = pred_name
out_df["pred_conf"] = pred_conf

os.makedirs(os.path.dirname(OUT_PREDS) or ".", exist_ok=True)
out_df.to_csv(OUT_PREDS, index=False)
print(f"Saved: {OUT_PREDS}")

# -------------------------------------------------------
# 6) Optional evaluation if NEW_CSV has ground truth labels
# -------------------------------------------------------
if label_col is not None and label_col in out_df.columns:
    y_true_str = out_df[label_col].astype(str).values
    # map unseen labels to -1
    y_true = np.array([label_map.get(v, -1) for v in y_true_str], dtype=np.int64)

    valid = (y_true >= 0) & (pred_id >= 0)
    if valid.sum() > 0:
        acc = accuracy_score(y_true[valid], pred_id[valid])
        print(f"Eval on {valid.sum()} labeled rows: accuracy={acc:.4f}")

        cm = confusion_matrix(y_true[valid], pred_id[valid], labels=np.arange(num_cls))
        print("Confusion matrix shape:", cm.shape)
    else:
        print("No valid labeled rows for evaluation (labels missing or unseen).")


[Training artifacts] in_dim=45, num_cls=28
[NEW] Total cells: 161867
[NEW] Built 544 mini-graphs across 8 region(s)
[Model] Loaded weights and set to eval()
Saved: /mnt/jwh83-data/Confetti/output/Redsea/Clustering/20260119_ns444_mesmer_afterREDSEA_gnn_predict.csv
